# CONFIGURATION

In [13]:
import os
from datetime import datetime

# -----------------------
# CONFIGURATION
# -----------------------
CONFIG = {
    # Dataset directories
    "train_dataset_dir": "data\\processed\\crackforest_augmented_50_15_35\\train",
    "val_dataset_dir": "data\\processed\\crackforest_augmented_50_15_35\\val",
    "test_dataset_dir": "data\\processed\\crackforest_augmented_50_15_35\\test",

    # Training parameters
    "epochs": 30,
    "batch_size": 32,
    "learning_rate": 0.0001,
    "img_size": 224,
    
    # Model
    "freeze_backbone": False,
    
    # Output
    "output_dir": "experiments_segmentation",
    
    # Checkpoints
    "resume_checkpoint": None,
    "test_checkpoint": None,
    "test_only": False,
    
    # TensorBoard
    "use_tensorboard": True
}

# Generate run ID and save directory
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = os.path.join(CONFIG["output_dir"], RUN_ID)
os.makedirs(SAVE_DIR, exist_ok=True)
log_dir = os.path.join(SAVE_DIR, "logs")
os.makedirs(log_dir, exist_ok=True)

print("Run ID:", RUN_ID)
print("Save Directory:", SAVE_DIR)

Run ID: 20260426_140749
Save Directory: experiments_segmentation\20260426_140749


# IMPORT and SETUP

In [14]:
# -----------------------
# IMPORTS
# -----------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

from PIL import Image
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import OrderedDict
import torch.nn.functional as F

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# DATASET CLASS

In [15]:
# -----------------------
# DATASET CLASS
# -----------------------
class CrackForestDataset(Dataset):
    def __init__(self, root_dir, img_size=224, transform=None):
        self.img_dir = os.path.join(root_dir, "Images")
        self.mask_dir = os.path.join(root_dir, "Masks")
        # Include only image files
        self.images = sorted([f for f in os.listdir(self.img_dir) 
                              if f.lower().endswith(('.jpg', '.png', '.bmp'))])
        self.img_size = img_size
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        base = os.path.splitext(img_name)[0]
        mask_name = base + "_label.PNG"

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Load image and mask
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize
        image = image.resize((self.img_size, self.img_size))
        mask = mask.resize((self.img_size, self.img_size))

        # To tensor
        image = transforms.ToTensor()(image)
        mask = torch.from_numpy(np.array(mask)/255.0).float().unsqueeze(0)

        return image, mask

# DATALOADER

In [16]:
# -----------------------
# DATALOADERS
# -----------------------

# Create datasets
train_ds = CrackForestDataset(CONFIG["train_dataset_dir"], img_size=CONFIG["img_size"])
val_ds = CrackForestDataset(CONFIG["val_dataset_dir"], img_size=CONFIG["img_size"])
test_ds = CrackForestDataset(CONFIG["test_dataset_dir"], img_size=CONFIG["img_size"])

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Test samples: {len(test_ds)}")

Train samples: 944, Val samples: 17, Test samples: 42


# LOAD MODEL

UNet with MobileNetV2 Backbone

In [17]:
from torchvision.models import mobilenet_v2

class UNetMobileNetV2(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.num_classes = num_classes

        # -----------------------
        # Encoder: MobileNetV2
        # -----------------------
        backbone = mobilenet_v2(pretrained=pretrained).features

        # Pick layers to use as skip connections
        self.enc0 = backbone[0:2]     # out_channels = 16
        self.enc1 = backbone[2:4]     # out_channels = 24
        self.enc2 = backbone[4:7]     # out_channels = 32
        self.enc3 = backbone[7:14]    # out_channels = 96
        self.enc4 = backbone[14:18]   # out_channels = 320

        # -----------------------
        # Decoder
        # -----------------------
        self.dec4 = nn.Conv2d(320, 96, kernel_size=3, padding=1)
        self.dec3 = nn.Conv2d(96 + 96, 32, kernel_size=3, padding=1)
        self.dec2 = nn.Conv2d(32 + 32, 24, kernel_size=3, padding=1)
        self.dec1 = nn.Conv2d(24 + 24, 16, kernel_size=3, padding=1)
        self.dec0 = nn.Conv2d(16 + 16, 16, kernel_size=3, padding=1)

        # Final segmentation layer
        self.final = nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        # -----------------------
        # Encoder
        # -----------------------
        e0 = self.enc0(x)  # 16 channels
        e1 = self.enc1(e0)  # 24
        e2 = self.enc2(e1)  # 32
        e3 = self.enc3(e2)  # 96
        e4 = self.enc4(e3)  # 320

        # -----------------------
        # Decoder
        # -----------------------
        d4 = F.interpolate(self.dec4(e4), size=e3.shape[2:], mode='bilinear', align_corners=False)
        d4 = torch.cat([d4, e3], dim=1)

        d3 = F.interpolate(self.dec3(d4), size=e2.shape[2:], mode='bilinear', align_corners=False)
        d3 = torch.cat([d3, e2], dim=1)

        d2 = F.interpolate(self.dec2(d3), size=e1.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, e1], dim=1)

        d1 = F.interpolate(self.dec1(d2), size=e0.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, e0], dim=1)

        d0 = F.interpolate(self.dec0(d1), size=x.shape[2:], mode='bilinear', align_corners=False)

        out = self.final(d0)
        return out

# Initialize model
model = UNetMobileNetV2(pretrained=True).to(device)
print(model)

# -----------------------
# LOSS FUNCTION
# -----------------------
criterion = nn.BCEWithLogitsLoss()  # For binary segmentation

# -----------------------
# OPTIMIZER
# -----------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"]
)

# Optional: learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

print("Loss, optimizer, and scheduler set up.")

UNetMobileNetV2(
  (enc0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
  )
  (enc1): Sequential(
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
         

# TRAINING

In [18]:
# TensorBoard
writer = None
if CONFIG["use_tensorboard"]:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir=log_dir)

best_val_loss = float('inf')

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss = 0.0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} - Train"):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} - Val"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # TensorBoard logging
    if writer:
        writer.add_scalar("Loss/Train", train_loss, epoch)
        writer.add_scalar("Loss/Val", val_loss, epoch)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = os.path.join(SAVE_DIR, "best_model.pth")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Saved best model checkpoint: {checkpoint_path}")

# Close TensorBoard
if writer:
    writer.close()

Epoch 1/30 - Val: 100%|██████████| 1/1 [00:00<00:00,  2.46it/s]


Epoch 1 | Train Loss: 0.5990 | Val Loss: 0.5545
Saved best model checkpoint: experiments_segmentation\20260426_140749\best_model.pth


Epoch 2/30 - Val: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]


Epoch 2 | Train Loss: 0.1319 | Val Loss: 0.0749
Saved best model checkpoint: experiments_segmentation\20260426_140749\best_model.pth


Epoch 3/30 - Val: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]


...

Epoch 28 | Train Loss: 0.0281 | Val Loss: 0.0293
Saved best model checkpoint: experiments_segmentation\20260426_140749\best_model.pth


Epoch 29/30 - Val: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]


Epoch 29 | Train Loss: 0.0279 | Val Loss: 0.0291
Saved best model checkpoint: experiments_segmentation\20260426_140749\best_model.pth


Epoch 30/30 - Val: 100%|██████████| 1/1 [00:00<00:00,  2.57it/s]

Epoch 30 | Train Loss: 0.0280 | Val Loss: 0.0288
Saved best model checkpoint: experiments_segmentation\20260426_140749\best_model.pth


# TESTING

In [19]:
# Load best model if test_only
if CONFIG["test_checkpoint"]:
    model.load_state_dict(torch.load(CONFIG["test_checkpoint"], map_location=device))
    model.to(device)
    print(f"Loaded checkpoint: {CONFIG['test_checkpoint']}")

all_preds = []
all_masks = []

model.eval()
with torch.no_grad():
    for imgs, masks in test_loader:  # replace test_loader with your DataLoader
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(imgs)
        
        # Apply sigmoid if your model doesn't already have it
        preds = torch.sigmoid(outputs)
        
        # Flatten to 1D arrays
        all_preds.append(preds.cpu().numpy().flatten())
        all_masks.append(masks.cpu().numpy().flatten())

# Concatenate all batches
all_preds_flat = np.concatenate(all_preds)
all_masks_flat = np.concatenate(all_masks)

# Binarize predictions at threshold 0.5
all_preds_flat_bin = (all_preds_flat > 0.5).astype(int)
all_masks_flat_bin = all_masks_flat.astype(int)

# ----- Metrics -----
# Confusion matrix
conf_matrix = confusion_matrix(all_masks_flat_bin, all_preds_flat_bin)
print("Confusion Matrix:")
print(conf_matrix)

# Dice score
def dice_score(y_true, y_pred, epsilon=1e-6):
    intersection = np.sum(y_true * y_pred)
    return (2 * intersection + epsilon) / (np.sum(y_true) + np.sum(y_pred) + epsilon)

# IoU score
def iou_score(y_true, y_pred, epsilon=1e-6):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + epsilon) / (union + epsilon)

dice = dice_score(all_masks_flat_bin, all_preds_flat_bin)
iou = iou_score(all_masks_flat_bin, all_preds_flat_bin)

print(f"Dice Score: {dice:.4f}")
print(f"IoU Score: {iou:.4f}")

# Save metrics to CSV
metrics_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "iou": iou,
    "dice": dice,
    **CONFIG
}])

metrics_path = os.path.join(SAVE_DIR, "metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print("Saved metrics to:", metrics_path)

# Optionally save confusion matrix figure
plt.figure(figsize=(5,5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"))
plt.close()
print("Saved confusion matrix figure.")

Confusion Matrix:
[[2072830   17745]
 [   8158    8659]]
Dice Score: 0.4007
IoU Score: 0.2505
Saved metrics to: experiments_segmentation\20260426_140749\metrics.csv
Saved confusion matrix figure.


# SUMMARY

In [20]:
# -----------------------
# SUMMARY CSV
# -----------------------
summary = {
    "run_id": RUN_ID,
    "iou": iou,
    "dice": dice,
    **CONFIG
}

summary_path = os.path.join(CONFIG["output_dir"], "summary_segmentation.csv")

if os.path.exists(summary_path):
    df_old = pd.read_csv(summary_path)
    df_new = pd.concat([df_old, pd.DataFrame([summary])], ignore_index=True)
else:
    df_new = pd.DataFrame([summary])

df_new.to_csv(summary_path, index=False)
print("Updated summary_segmentation.csv")

# -----------------------
# SAVE FINAL MODEL
# -----------------------
final_model_path = os.path.join(SAVE_DIR, "final_model.pth")
torch.save(model.state_dict(), final_model_path)
print("Saved final model weights:", final_model_path)

Updated summary_segmentation.csv
Saved final model weights: experiments_segmentation\20260426_140749\final_model.pth
